# Lab 2 — Partea C: Optimizarea hiperparametrilor și Bonus

Această secțiune se concentrează pe îmbunătățirea performanței modelului prin ajustarea hiperparametrilor. Vom folosi strategia optimă de pregătire a datelor stabilită în Partea B (codificare ordinală, cele 3 *feature*-uri inginerite și ponderi balansate pentru clase).

Vom parcurge următorii pași:
1. **Tuning manual:** Ajustarea intuitivă a hiperparametrilor cheie (`max_depth`, `learning_rate`, `n_estimators`) și analizarea compromisului *bias-variance*.
2. **Optimizare Bayesiană cu Optuna (`TPESampler`):** Căutarea automată a celei mai bune combinații de hiperparametri, generarea graficelor de optimizare și salvarea pașilor intermediari.
3. **Bonus:** Implementarea unui al doilea algoritm de tip *boosted trees* (LightGBM) pe exact aceleași date și compararea rezultatelor.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import StratifiedKFold

# Importăm funcțiile gata pregătite din lab2_utils.py
from lab2_utils import get_final_data, get_split, balanced_weights, cv_macro_f1

# 1. Încărcăm datele finale (conțin deja cele 3 feature-uri noi și codificarea ordinală)
X, y = get_final_data()

# 2. Facem împărțirea fixă train/test (80/20, stratificată)
X_train, X_test, y_train, y_test = get_split(X, y)

# 3. Calculăm ponderile balansate pentru clase
sample_weights_train = balanced_weights(y_train)

print("Date încărcate cu succes prin lab2_utils!")
print("Forma matricei de antrenare (X_train):", X_train.shape)
print("Distribuția claselor în y_train:\n", y_train.value_counts().sort_index())

Date încărcate cu succes prin lab2_utils!
Forma matricei de antrenare (X_train): (2344, 253)
Distribuția claselor în y_train:
 PriceClass
0    189
1    407
2    830
3    549
4    318
5     51
Name: count, dtype: int64


2344 = numărul de case folosite la antrenare (80% din 2930).
253 = numărul de caracteristici (coloane) pe care le vede modelul pentru fiecare casă: suprafață, an, calitate etc. După pregătirea Dianei.

Clasa	Preț	Număr de case	Procent

0	< 100k	189	8%

1	100–130k	407	17%

2	130–180k	830	35%

3	180–250k	549	23%

4	250–400k	318	14%

5	≥ 400k	51	2%

## 2. Optimizare Bayesiană cu Optuna (`TPESampler`)

Pentru a găsi combinația ideală de hiperparametri fără o căutare oarbă (*grid search*), folosim librăria `optuna` împreună cu **`TPESampler`** (Tree-structured Parzen Estimator), algoritm care corespunde optimizării bayesiene.

Optimizăm următorii parametri:
- `max_depth`: între 3 și 9
- `learning_rate`: între 0.01 și 0.2 (scară logaritmică)
- `n_estimators`: între 50 și 300
- `subsample` și `colsample_bytree`: între 0.6 și 1.0
- `min_child_weight`: între 1 și 5
- `gamma`: între 0.0 și 1.0

In [2]:
import optuna
from sklearn.model_selection import StratifiedKFold

# Ascundem mesajele repetitive de logare Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }
    
    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Folosim funcția cv_macro_f1 din lab2_utils care aplică automat și sample weights
    scores = cv_macro_f1(model, X_train, y_train, cv=cv, weights=True)
    return scores.mean()

# Inițializăm TPESampler conform cerinței din laborator[cite: 11]
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)

# Rulăm optimizarea (30 de trial-uri)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Cel mai bun scor Macro-F1 găsit:", round(study.best_value, 4))
print("\nCei mai buni hiperparametri găsiți:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/30 [00:00<?, ?it/s]

Cel mai bun scor Macro-F1 găsit: 0.7655

Cei mai buni hiperparametri găsiți:
  max_depth: 7
  learning_rate: 0.07156209339149121
  n_estimators: 163
  subsample: 0.9773299726247243
  colsample_bytree: 0.8243250837845572
  min_child_weight: 3
  gamma: 0.06967112182086788


**Cum evaluăm fiecare combinație de hiperparametri:**

Fiecare încercare a lui Optuna e notată prin **5-fold cross-validation stratificat**, doar pe datele de antrenare (2344 de case). Casele sunt împărțite în 5 grupe egale, cu aceleași proporții ale claselor de preț. Modelul se antrenează de 5 ori: de fiecare dată învață pe 4 grupe și e evaluat pe a cincea. Scorul unei încercări e media celor 5 valori de macro-F1.

Așa, fiecare casă e folosită o dată pentru evaluare, iar scorul nu depinde de o singură împărțire. Ponderile balansate sunt calculate separat pentru fiecare fold. Setul de test nu e folosit în această etapă, ci păstrat pentru evaluarea finală.

După 30 de încercări (~9 minute),
 Optuna a găsit cea mai bună combinație la încercarea 20,
  cu macro-F1 0.7655 la CV.
  
   Modelul ales învață mai încet (rata 0.07 cu 163 de arbori) și are 
   frâne împotriva memorării (min_child_weight 3, gamma, eșantionarea coloanelor)

   
UNDE AM OBS MEMORARE:

Pe casele de antrenare, macro-F1 = 1.000, adică perfect.
Pe casele de test, macro-F1 = 0.748.

### Vizualizarea procesului de optimizare și salvarea studiului

Generăm graficele pentru istoricul optimizării și importanța hiperparametrilor, salvând totodată studiul Optuna pe disc.

In [ ]:
import joblib
from optuna.visualization import plot_optimization_history, plot_param_importances

# 1. Generăm graficele interac tive Plotly pentru Optuna
fig_hist = plot_optimization_history(study)
fig_hist.update_layout(title="Istoricul optimizării Optuna (Macro-F1)")
fig_hist.show()

fig_imp = plot_param_importances(study)
fig_imp.update_layout(title="Importanța hiperparametrilor în optimizare")
fig_imp.show()

# 2. Salvăm studiul pe disc (pașii intermediari)
joblib.dump(study, "optuna_ames_study.pkl")
print("Studiul Optuna a fost salvat cu succes în fișierul 'optuna_ames_study.pkl'.")

Studiul Optuna a fost salvat cu succes în fișierul 'optuna_ames_study.pkl'.


max_depth   -    arbori puțin mai adânci: fiecare poate pune până la 7 întrebări la rând

learning_rate	-	fiecare arbore corectează de ~4 ori mai puțin agresiv

n_estimators	-	mai mulți arbori, ca să compenseze pașii mai mici

subsample	-	fiecare arbore vede ~98% din case, deci aproape toate

colsample_bytree	-	fiecare arbore vede 82% din coloane, alese la întâmplare

min_child_weight	-	o frunză trebuie să aibă „greutate” de cel puțin 3 (cu ponderi, o casă rară cântărește mai mult)

gamma	-	o împărțire se face doar dacă reduce eroarea măcar puțin

Primul grafic arată că Optuna a găsit zona bună după ~10 încercări și s-a stabilizat după 20, deci căutarea a fost suficientă, iar câștigul față de setările implicite e mic. Al doilea arată că a contat cel mai mult subsample, adică reducerea memorării, ceea ce confirmă că problema principală a modelului era overfitting-ul.

Mai multe combinații dau practic același scor, iar diferențele sunt mai mici decât zgomotul cross-validation-ului. Când am reverificat cu CV repetat, scorul real a fost ~0.755, nu 0.7655. Contează ideea generală, adică învățare mai lentă și mai multă regularizare, nu valorile exacte.

## 3. Tuning manual de hiperparametri și compromisul Bias-Variance

Compromisul bias–variance înseamnă că un model prea simplu nu învață regulile (bias), iar unul prea complex le memorează și pe cele întâmplătoare (variance). Modelul nostru avea variance mare: 1.0 pe antrenare și ~0.75 la CV. De aceea hiperparametrii care au contat sunt cei care frânează memorarea

Înainte de optimizarea automată, testăm manual câțiva hiperparametri pentru a observa cum se modifică rezultatele și cum influențează aceștia compromisul *bias-variance*:
- **`max_depth`**: O adâncime prea mică duce la *underfitting* (bias mare), iar una prea mare duce la *overfitting* (varianță mare).
- **`learning_rate`**: Rata de învățare a modelului.

In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Testăm manual câteva seturi de hiperparametri de bază
manual_configs = {
    "Baseline (default)": XGBClassifier(random_state=42),
    "Adâncime mică (max_depth=3)": XGBClassifier(max_depth=3, random_state=42),
    "Adâncime mare (max_depth=8, lr=0.2)": XGBClassifier(max_depth=8, learning_rate=0.2, random_state=42),
    "Rată învățare mică (lr=0.05, n=200)": XGBClassifier(learning_rate=0.05, n_estimators=200, random_state=42),
}

manual_results = []
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in manual_configs.items():
    # Nota pe casele noi (cross-validation)
    scores = cv_macro_f1(model, X_train, y_train, cv=cv5, weights=True)

    # Nota pe casele pe care modelul le-a văzut (antrenare)
    model.fit(X_train, y_train, sample_weight=sample_weights_train)
    train_f1 = f1_score(y_train, model.predict(X_train), average="macro")

    manual_results.append({
        "Configurație": name,
        "Train Macro-F1": train_f1,
        "CV Macro-F1": scores.mean(),
        "Std": scores.std(),
        "Diferență (Train - CV)": train_f1 - scores.mean(),
    })

df_manual = pd.DataFrame(manual_results).sort_values("CV Macro-F1", ascending=False).round(4)
display(df_manual)

,Configurație,Train Macro-F1,CV Macro-F1,Std,Diferență (Train - CV)
2,"Adâncime mare (max_depth=8, lr=0.2)",1.0000,0.7535,0.0161,0.2465
1,Adâncime mică (max_depth=3),0.9779,0.7467,0.0204,0.2311
0,Baseline (default),1.0000,0.7462,0.0185,0.2538
3,"Rată învățare mică (lr=0.05, n=200)",0.9920,0.7293,0.0191,0.2627


- Train Macro-F1: cât de bine ghicește casele pe care le-a văzut la învățare.
- CV Macro-F1: cât de bine ghicește casele pe care nu le-a văzut. Asta e nota care contează.
- Diferența: Train minus CV. Arată cât memorează.
- Std: cât de nesigură e nota de la CV (~0.02).

**Interpretare:**

Am adăugat și scorul pe datele de antrenare, ca să vedem cât de mult memorează modelul.

La toate configurațiile scorul pe antrenare e aproape 1 (0.98–1.00), iar la cross-validation e în jur de 0.73–0.75. Diferența de aproximativ 0.25 arată că modelul memorează datele de antrenare, oricum l-am seta. Deci problema noastră e varianța mare (overfitting), nu bias-ul.

Cu `max_depth=3` modelul memorează puțin mai puțin (diferența scade la 0.23), dar scorul la CV rămâne la fel. Cu `max_depth=8` și `lr=0.2` am obținut cel mai bun scor la CV (0.7535), dar e doar cu 0.007 peste varianta implicită, mai puțin decât deviația standard (~0.02), așa că nu putem spune sigur că e mai bun. În plus, am schimbat doi parametri deodată.

Cu `lr=0.05` și 200 de arbori modelul învață bine datele de antrenare (0.992), dar are cel mai slab scor la CV (0.729). Pașii sunt mai mici, iar 200 de arbori nu sunt suficienți ca modelul să generalizeze bine, deci rata de învățare trebuie aleasă împreună cu numărul de arbori.

Diferențele dintre configurații sunt mici, așa că am trecut la o căutare automată cu Optuna. Pentru că modelul memorează mult, ne așteptăm ca parametrii care reduc overfitting-ul să conteze cel mai mult.

## 4. BONUS: Implementarea și optimizarea LightGBM

Implementăm un al doilea algoritm bazat pe arbori (*boosted trees*) — **LightGBM** (`LGBMClassifier`) — pe exact aceleași date și caracteristici, comparând rezultatele finale cu XGBoost.

In [6]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

# Antrenăm LightGBM folosind aceleași date și ponderi balansate
lgb_model = LGBMClassifier(random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)

# Evaluăm prin 5-fold CV
lgb_scores = cv_macro_f1(lgb_model, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), weights=True)
print(f"LightGBM 5-fold CV Macro-F1: {lgb_scores.mean():.4f} +/- {lgb_scores.std():.4f}")

# Comparație directă pe setul de test între XGBoost optimizat (prin Optuna) și LightGBM
best_xgb = XGBClassifier(**study.best_params, random_state=42)
best_xgb.fit(X_train, y_train, sample_weight=sample_weights_train)

xgb_test_pred = best_xgb.predict(X_test)
lgb_test_pred = lgb_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Model": ["XGBoost (Optimizat Optuna)", "LightGBM (Baseline)"],
    "Test Macro-F1": [
        f1_score(y_test, xgb_test_pred, average="macro"),
        f1_score(y_test, lgb_test_pred, average="macro")
    ]
})
comparison_df.round(4)

LightGBM 5-fold CV Macro-F1: 0.7492 +/- 0.0166


,Model,Test Macro-F1
0,XGBoost (Optimizat Optuna),0.7551
1,LightGBM (Baseline),0.7398


In [7]:
from lightgbm import LGBMClassifier

def objective_lgbm(trial):
    params = {
        "num_leaves":        trial.suggest_int("num_leaves", 8, 64),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 50, 400),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model = LGBMClassifier(**params, random_state=42, verbose=-1)
    return cv_macro_f1(model, X_train, y_train, cv=cv5, weights=True).mean()

study_lgbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                 storage="sqlite:///optuna_lgbm.db", study_name="lgbm", load_if_exists=True)
study_lgbm.optimize(objective_lgbm, n_trials=30, show_progress_bar=True)

print("Cel mai bun scor LightGBM:", round(study_lgbm.best_value, 4))
print(study_lgbm.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Cel mai bun scor LightGBM: 0.766
{'num_leaves': 39, 'learning_rate': 0.017398074711291726, 'n_estimators': 390, 'min_child_samples': 32, 'colsample_bytree': 0.9636993649385135, 'reg_lambda': 3.7958531426706403}


In [8]:
best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=42, verbose=-1)

comparatie = pd.DataFrame({
    "Model": ["XGBoost optimizat", "LightGBM optimizat"],
    "CV Macro-F1": [
        cv_macro_f1(XGBClassifier(**study.best_params, random_state=42), X_train, y_train, cv=cv5).mean(),
        cv_macro_f1(best_lgbm, X_train, y_train, cv=cv5).mean(),
    ],
})
comparatie.round(4)

,Model,CV Macro-F1
0,XGBoost optimizat,0.7655
1,LightGBM optimizat,0.7660


### Rezultatele optimizării LightGBM

Am optimizat LightGBM cu Optuna în aceleași condiții ca XGBoost: aceleași date, aceleași ponderi și același 5-fold cross-validation.

**Cei mai buni hiperparametri găsiți:**

| Hiperparametru | Valoare | Ce înseamnă |
|---|---|---|
| `learning_rate` | 0.017 | modelul învață foarte încet |
| `n_estimators` | 390 | mulți arbori, ca să compenseze pașii mici |
| `num_leaves` | 39 | complexitatea unui arbore |
| `min_child_samples` | 32 | fiecare frunză are cel puțin 32 de case, ca modelul să nu memoreze |
| `colsample_bytree` | 0.96 | fiecare arbore vede aproape toate coloanele |

Ca și la XGBoost, cel mai bun model **învață încet și e frânat ca să nu memoreze**. Numărul de arbori a ieșit aproape de limita intervalului (400), deci intervalul ar putea fi lărgit.

### Comparația XGBoost vs LightGBM

| Model | CV Macro-F1 |
|---|---|
| XGBoost optimizat | 0.7655 |
| LightGBM optimizat | 0.7660 |

După optimizare, cele două modele au **practic același scor**. Diferența (0.0005) e mult mai mică decât deviația standard a cross-validation-ului (~0.02), deci nu putem spune că unul e mai bun.

Comparația anterioară pe setul de test (XGBoost 0.7551 vs LightGBM 0.7398) nu era corectă, pentru că LightGBM nu fusese optimizat.